# 02 – Tiền xử lý dữ liệu Spotify songs (cho hierarchical clustering)

**Đầu vào** (đều là đầu ra của `01_data_inspection`):
- `../data/raw/spotify_songs.csv` – dữ liệu gốc
- `../data/processed/spotify_songs_preprocessed.csv` – bản mà các ô *ngụy trang* đã được đổi thành NaN
- `../data/processed/dq_report/` – báo cáo chất lượng dữ liệu

**Các bước (theo thứ tự):**
1. Xóa toàn bộ các dòng của `track_id` có giá trị thiếu / ngụy trang. **Giữ lại** (không xóa) các track bị đánh giá là:
   - ≥ nửa số đặc trưng audio bằng 0 trong cùng 1 dòng
   - `track_popularity = 0`
   - `instrumentalness` (giá trị 0 / spike)
2. Xóa toàn bộ các dòng của `track_id` có `key = -1`.
3. Xóa trùng: (3a) trùng `(track_id, playlist_id)`, (3b) trùng `track_id` – giữ dòng đầu tiên → mỗi `track_id` đúng 1 dòng.

**Cách xử lý từng luật trong dq_report**

| Luật trong dq_report | Xử lý |
|---|---|
| Chuỗi placeholder (`'NA'`, `'?'`) ở `track_name` / `track_artist` / `track_album_name` | Xóa track (bước 1) |
| `loudness` ngoài [-60, 0] | Xóa track (bước 1) |
| `tempo <= 0` | Xóa track (bước 1), trừ khi dòng đó thuộc nhóm "≥ nửa audio = 0" |
| `track_album_release_date = 1970-01-01` (epoch) | Xóa track (bước 1) |
| ≥ nửa đặc trưng audio = 0 | **Giữ** |
| `track_popularity = 0` | **Giữ** |
| `instrumentalness` spike tại 0 | **Giữ** |
| `duration_ms` ngoài [30 giây, 20 phút] (2 dòng) | Giữ (đổi `DROP_DURATION_OUTLIERS = True` để xóa) |
| Ngày phát hành chỉ có năm / năm-tháng / 01-01 | Giữ (chỉ là độ chính xác thấp) |
| `key = -1` | Xóa track (bước 2) |
| Trùng `track_id`, trùng `(track_id, playlist_id)` | Bước 3 |
| Trùng tiềm ẩn (cùng tên + nghệ sĩ nhưng khác `track_id`, cùng audio fingerprint) | Chưa xử lý (ngoài yêu cầu) |

In [15]:
from pathlib import Path

import pandas as pd
from IPython.display import display

DATA_DIR   = Path("../data")
RAW_CSV    = DATA_DIR / "raw" / "spotify_songs.csv"
PRE_CSV    = DATA_DIR / "processed" / "spotify_songs_preprocessed.csv"   # ô ngụy trang -> NaN (từ 01)
DQ_DIR     = DATA_DIR / "processed" / "dq_report"
OUT_DIR    = DATA_DIR / "processed"
OUT_CSV    = OUT_DIR / "spotify_songs_clean.csv"
REPORT_DIR = OUT_DIR / "preprocess_report"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

STR_COLS = ["track_id", "track_name", "track_artist", "track_album_id",
            "track_album_name", "track_album_release_date", "playlist_name",
            "playlist_id", "playlist_genre", "playlist_subgenre"]
AUDIO_COLS = ["danceability", "energy", "key", "loudness", "mode", "speechiness",
              "acousticness", "instrumentalness", "liveness", "valence",
              "tempo", "duration_ms"]

# Luật "đáng ngờ" trong dq_report -- mặc định KHÔNG xóa
DROP_DURATION_OUTLIERS = False     # duration_ms ngoài [30 giây, 20 phút]

In [16]:
def read_csv_keep_na(path):
    """Giống load() của 01: chỉ ô rỗng là NaN, giữ nguyên 'NA', 'null'..."""
    df = pd.read_csv(path, keep_default_na=False, na_values=[""], low_memory=False,
                     dtype={c: "string" for c in STR_COLS})
    df.columns = df.columns.str.strip()
    return df

raw = read_csv_keep_na(RAW_CSV)
pre = read_csv_keep_na(PRE_CSV)

assert len(raw) == len(pre) and list(raw.columns) == list(pre.columns), \
    "raw và preprocessed không khớp -> chạy lại 01_data_inspection"

for c in STR_COLS:
    raw[c] = raw[c].astype("string").str.strip()

print(f"Dữ liệu gốc: {len(raw):,} dòng, {raw['track_id'].nunique():,} track_id")
print("\nDữ liệu ngụy trang (dq_report/2_disguised_missing.csv):")
display(pd.read_csv(DQ_DIR / "2_disguised_missing.csv")[["column", "rule", "n_rows"]])
print("Trùng lặp (dq_report/dup_summary.csv):")
display(pd.read_csv(DQ_DIR / "dup_summary.csv"))

Dữ liệu gốc: 32,833 dòng, 28,356 track_id

Dữ liệu ngụy trang (dq_report/2_disguised_missing.csv):


,column,rule,n_rows
0,track_name,chuỗi placeholder (NA/null/unknown/-/0/rỗng...),6
1,track_artist,chuỗi placeholder (NA/null/unknown/-/0/rỗng...),5
2,track_album_name,chuỗi placeholder (NA/null/unknown/-/0/rỗng...),9
3,loudness,"ngoài khoảng [-60, 0]",6
4,tempo,tempo <= 0,1
5,duration_ms,"duration ngoài [30 giây, 20 phút] (đáng ngờ)",2
6,track_popularity,popularity = 0 (có thể là thiếu hoặc bài chưa ...,2703
7,(nhiều cột audio),≥ nửa số đặc trưng audio bằng 0 trong cùng 1 dòng,1
8,track_popularity,một giá trị chiếm ≥ 5% số dòng (spike),2703
9,instrumentalness,một giá trị chiếm ≥ 5% số dòng (spike),12089


Trùng lặp (dq_report/dup_summary.csv):


,loại,khóa,số dòng thuộc nhóm trùng,số dòng thừa (giữ 1)
0,Trùng hoàn toàn (mọi cột),(all),0,0
1,Trùng khóa nghiệp vụ,track_id,7643,4477
2,Trùng khóa nghiệp vụ,track_id + playlist_id,1114,582
3,Trùng khóa nghiệp vụ,track_name + track_artist,10666,6603
4,Trùng khóa nghiệp vụ,track_name + track_artist + track_album_name,8829,5259


In [17]:
def audio_zero_rows(df):
    """Dòng có >= nửa số đặc trưng audio bằng 0 (cùng luật với 01)."""
    return (df[AUDIO_COLS].fillna(1) == 0).sum(axis=1) >= max(3, len(AUDIO_COLS) // 2)


def remove_tracks(df, bad_ids, drop_null_id=True):
    """Xóa mọi dòng thuộc các track_id trong bad_ids (và dòng thiếu track_id)."""
    drop = df["track_id"].isin(set(bad_ids))
    if drop_null_id:
        drop |= df["track_id"].isna()
    return df[~drop]


log, removed = [], []

def snapshot(step, df, before):
    log.append({
        "step": step,
        "rows_removed": before[0] - len(df),
        "track_ids_removed": before[1] - df["track_id"].nunique(),
        "rows_after": len(df),
        "track_ids_after": df["track_id"].nunique(),
    })
    print(f"{step:<42} -{before[0] - len(df):>6,} dòng | còn {len(df):>7,} dòng, "
          f"{df['track_id'].nunique():>7,} track_id")


df = raw.copy()
log.append({"step": "0. Dữ liệu gốc", "rows_removed": 0, "track_ids_removed": 0,
            "rows_after": len(df), "track_ids_after": df["track_id"].nunique()})
print(f"{'0. Dữ liệu gốc':<42} {'':>8} dòng | còn {len(df):>7,} dòng, {df['track_id'].nunique():>7,} track_id")

0. Dữ liệu gốc                                      dòng | còn  32,833 dòng,  28,356 track_id


## Bước 1. Xóa track_id có giá trị thiếu / ngụy trang

`pre.isna()` = NaN gốc + các ô ngụy trang mà `01` đã đổi thành NaN (placeholder, loudness ngoài khoảng, tempo ≤ 0, ngày 1970-01-01...).
Các luật *mềm* (popularity = 0, instrumentalness spike, nửa audio = 0) không nằm trong mask nên các track đó được giữ.
Ngoại lệ: dòng "nửa audio = 0" thường kéo theo `tempo <= 0` (nằm trong mask) → bảo vệ các cột audio của những dòng này để track không bị xóa nhầm.

In [18]:
az = audio_zero_rows(df)

protect = pd.DataFrame(False, index=df.index, columns=df.columns)
protect.loc[az, AUDIO_COLS] = True

bad_cells = pre.isna() & ~protect

if DROP_DURATION_OUTLIERS:
    d = df["duration_ms"]
    bad_cells["duration_ms"] |= ((d < 30_000) | (d > 1_200_000)) & ~protect["duration_ms"]

row_bad = bad_cells.any(axis=1) & df["track_id"].notna()
bad_ids = df.loc[row_bad, "track_id"].unique()

why = bad_cells[row_bad].groupby(df.loc[row_bad, "track_id"]).any()
for tid, r in why.iterrows():
    removed.append({"track_id": tid, "step": 1,
                    "reason": "thiếu/ngụy trang ở: " + ", ".join(r.index[r])})
n_null_id = int(df["track_id"].isna().sum())
if n_null_id:
    removed.append({"track_id": "(thiếu track_id)", "step": 1,
                    "reason": f"{n_null_id} dòng không có track_id"})

print(f"{int(row_bad.sum())} dòng bị đánh dấu -> {len(bad_ids)} track_id sẽ bị xóa")
print(f"Giữ lại {int(az.sum())} dòng 'nửa audio = 0' nếu chúng không thiếu ở cột khác")

before = (len(df), df["track_id"].nunique())
df = remove_tracks(df, bad_ids)
snapshot("1. Xóa track thiếu/ngụy trang", df, before)

21 dòng bị đánh dấu -> 17 track_id sẽ bị xóa
Giữ lại 1 dòng 'nửa audio = 0' nếu chúng không thiếu ở cột khác
1. Xóa track thiếu/ngụy trang              -    21 dòng | còn  32,812 dòng,  28,339 track_id


## Bước 2. Xóa track_id có key = -1

In [19]:
key_ids = df.loc[df["key"] == -1, "track_id"].unique()
removed += [{"track_id": t, "step": 2, "reason": "key = -1"} for t in key_ids]

before = (len(df), df["track_id"].nunique())
df = remove_tracks(df, key_ids, drop_null_id=False)
snapshot("2. Xóa track có key = -1", df, before)

2. Xóa track có key = -1                   -     0 dòng | còn  32,812 dòng,  28,339 track_id


## Bước 3. Xóa trùng

Theo dq_report, các `track_id` lặp chỉ khác nhau ở cột `playlist_*`; các cột audio / popularity phải giống nhau → giữ dòng đầu tiên là an toàn (ô kiểm tra bên dưới xác nhận lại).

In [20]:
n_multi_genre = int((df.groupby("track_id")["playlist_genre"].nunique() > 1).sum())

before = (len(df), df["track_id"].nunique())
df = df.drop_duplicates(["track_id", "playlist_id"], keep="first")
snapshot("3a. Xóa trùng (track_id, playlist_id)", df, before)

# kiểm tra: cùng track_id có bị lệch audio / popularity không?
check_cols = ["track_popularity"] + AUDIO_COLS
conflict = int(df.groupby("track_id")[check_cols].nunique(dropna=False).gt(1).any(axis=1).sum())
print(f"   track_id có audio/popularity khác nhau giữa các dòng: {conflict}")

before = (len(df), df["track_id"].nunique())
df = df.drop_duplicates("track_id", keep="first")
snapshot("3b. Xóa trùng track_id", df, before)
print(f"   ({n_multi_genre:,} track_id xuất hiện ở > 1 playlist_genre; chỉ giữ genre của dòng đầu tiên)")

3a. Xóa trùng (track_id, playlist_id)      -   582 dòng | còn  32,230 dòng,  28,339 track_id
   track_id có audio/popularity khác nhau giữa các dòng: 0
3b. Xóa trùng track_id                     - 3,891 dòng | còn  28,339 dòng,  28,339 track_id
   (1,685 track_id xuất hiện ở > 1 playlist_genre; chỉ giữ genre của dòng đầu tiên)


## Hoàn thiện & xuất

In [21]:
df = df.reset_index(drop=True)
for c in ["key", "mode"]:
    if df[c].notna().all():
        df[c] = df[c].round().astype(int)

assert df["track_id"].is_unique, "track_id vẫn còn trùng"
assert not df["track_id"].isna().any(), "còn track_id bị thiếu"
print(f"Còn NaN trong dữ liệu cuối: {int(df.isna().sum().sum())} ô")
print(f"Kết quả: {len(df):,} dòng, {df['track_id'].nunique():,} track_id, {df.shape[1]} cột")

df.to_csv(OUT_CSV, index=False, encoding="utf-8")
pd.DataFrame(removed).to_csv(REPORT_DIR / "removed_tracks.csv", index=False, encoding="utf-8-sig")
pd.DataFrame(log).to_csv(REPORT_DIR / "preprocessing_log.csv", index=False, encoding="utf-8-sig")
print(f"Đã xuất: {OUT_CSV.resolve()}")
print(f"Log:     {REPORT_DIR.resolve()}")

display(pd.DataFrame(log))

Còn NaN trong dữ liệu cuối: 0 ô
Kết quả: 28,339 dòng, 28,339 track_id, 23 cột
Đã xuất: C:\Users\PC\OneDrive\Documents\Personal_Archive\01_University\Semester_03\CS115_Mathematics_for_Computer_science\05_Project\Final_term_project\Spotify-mood-clustering\data\processed\spotify_songs_clean.csv
Log:     C:\Users\PC\OneDrive\Documents\Personal_Archive\01_University\Semester_03\CS115_Mathematics_for_Computer_science\05_Project\Final_term_project\Spotify-mood-clustering\data\processed\preprocess_report


,step,rows_removed,track_ids_removed,rows_after,track_ids_after
0,0. Dữ liệu gốc,0,0,32833,28356
1,1. Xóa track thiếu/ngụy trang,21,17,32812,28339
2,2. Xóa track có key = -1,0,0,32812,28339
3,"3a. Xóa trùng (track_id, playlist_id)",582,0,32230,28339
4,3b. Xóa trùng track_id,3891,0,28339,28339
